# Power Plant Output Prediction

**Regression Project 57**

Full pipeline: EDA → preprocessing → baseline → LazyPredict → FLAML → top-3 evaluation.

## 2. Project Overview

Predict the net hourly electrical energy output (PE) of a Combined Cycle Power Plant
using ambient environmental variables: temperature, pressure, humidity, and exhaust vacuum.

This UCI dataset contains 9,568 data points collected over 6 years from a real power plant.
It's an excellent regression problem because the physics relationship between ambient
conditions and power output is well-understood, making the results interpretable.

## 3. Learning Objectives

1. Build a regression pipeline for industrial energy output prediction
2. Work with a clean, well-understood UCI dataset
3. Understand thermodynamic relationships in power generation
4. Compare linear vs non-linear models on physics-driven data
5. Evaluate with R², RMSE, MAE and residual analysis
6. Achieve high R² on a well-behaved regression problem

## 4. Problem Statement

> Given ambient conditions (temperature, pressure, humidity, vacuum), predict the **net hourly
> electrical energy output** (PE, in MW) of a combined cycle power plant.

This is a **regression** task. Primary metric: **R²**; secondary: RMSE, MAE.

## 5. Why This Project Matters

| Stakeholder | Impact |
|---|---|
| Power plant operators | Optimize generation scheduling |
| Grid dispatchers | Forecast available capacity |
| Energy traders | Price forecasting |
| Environmental engineers | Emissions estimation |
| ML learners | Clean, physics-based regression problem |

## 6. Dataset Overview

| Property | Value |
|---|---|
| Source | Kaggle: `andonians/random-linear-regression` (UCI CCPP) |
| Records | 9,568 |
| Features | 4 (AT, V, AP, RH) |
| Target | `PE` — net hourly electrical energy output (MW) |
| AT | Ambient Temperature (°C) |
| V | Exhaust Vacuum (cm Hg) |
| AP | Ambient Pressure (millibar) |
| RH | Relative Humidity (%) |

## 7. Dataset Source and License Notes

- **Kaggle**: [andonians/random-linear-regression](https://www.kaggle.com/datasets/andonians/random-linear-regression)
- Originally from UCI ML Repository (Tüfekci, 2014)
- Combined Cycle Power Plant dataset
- License: CC0 / Public Domain
- Downloaded via `kagglehub` at runtime

## 8. Environment Setup

In [ ]:
import subprocess, sys, importlib
for pkg in ['lazypredict', 'flaml', 'kagglehub', 'xgboost', 'lightgbm']:
    try: importlib.import_module(pkg)
    except ImportError: subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', pkg])
print('Environment ready.')

## 9. Imports

In [ ]:
import os, warnings, glob
import numpy as np, pandas as pd
import matplotlib.pyplot as plt, seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.dummy import DummyRegressor
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from lazypredict.Supervised import LazyRegressor
from flaml import AutoML
warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid')
SEED = 42
print('All imports loaded.')

## 10. Configuration / Constants

In [ ]:
KAGGLE_SLUG = 'andonians/random-linear-regression'
TARGET = 'PE'
TEST_SIZE = 0.15
VAL_SIZE = 0.15
FLAML_BUDGET = 120
MAX_ROWS = 50000

## 11. Dataset Download or Loading

Download the Combined Cycle Power Plant dataset via `kagglehub`.

In [ ]:
import kagglehub
try:
    path = kagglehub.dataset_download(KAGGLE_SLUG)
    print(f'Downloaded to: {path}')
except Exception as e:
    raise RuntimeError(
        f'Dataset download failed: {e}\n'
        'Ensure KAGGLE_API_TOKEN is set. '
        'Fallback: download from https://www.kaggle.com/datasets/andonians/random-linear-regression'
    )
all_files = glob.glob(os.path.join(path, '**', '*.*'), recursive=True)
csv_files = [f for f in all_files if f.endswith('.csv')]
xlsx_files = [f for f in all_files if f.endswith('.xlsx')]
# Try CSV first, then XLSX
if csv_files:
    df_raw = pd.read_csv(sorted(csv_files, key=os.path.getsize, reverse=True)[0])
elif xlsx_files:
    df_raw = pd.read_excel(sorted(xlsx_files, key=os.path.getsize, reverse=True)[0])
else:
    raise FileNotFoundError(f'No CSV/XLSX found in {path}')
print(f'Shape: {df_raw.shape}')
print(f'Columns: {df_raw.columns.tolist()}')
df_raw.head()

## 12. Data Validation Checks

Verify the target column exists and check data quality.

In [ ]:
df = df_raw.copy()
# The dataset may have AT, V, AP, RH, PE columns
# or may need column identification
if TARGET not in df.columns:
    # Try to find a suitable target column
    candidates = [c for c in df.columns if c.upper() in ['PE', 'EP', 'ENERGY']]
    if candidates:
        TARGET_USED = candidates[0]
    else:
        # Use last numeric column as target
        TARGET_USED = df.select_dtypes(include=[np.number]).columns[-1]
    print(f'Target column mapped to: {TARGET_USED}')
else:
    TARGET_USED = TARGET

print(f'Missing values:\n{df.isnull().sum()[df.isnull().sum() > 0]}')
print(f'Duplicated rows: {df.duplicated().sum()}')
df = df.dropna(subset=[TARGET_USED]).drop_duplicates().reset_index(drop=True)
if len(df) > MAX_ROWS:
    df = df.sample(MAX_ROWS, random_state=SEED).reset_index(drop=True)
TARGET = TARGET_USED
print(f'Shape: {df.shape}')
df.dtypes

## 13. Exploratory Data Analysis

Only 4 features — we can explore each thoroughly.

In [ ]:
df.describe()

In [ ]:
num_feats = [c for c in df.select_dtypes(include=[np.number]).columns if c != TARGET]
fig, axes = plt.subplots(2, 2, figsize=(12, 8))
for ax, col in zip(axes.flat, num_feats[:4]):
    df[col].hist(bins=30, ax=ax, alpha=0.7, color='steelblue')
    ax.set_title(col)
plt.tight_layout(); plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(df.select_dtypes(include=[np.number]).corr(), annot=True, fmt='.2f',
            cmap='coolwarm', center=0, ax=ax)
ax.set_title('Correlation Heatmap')
plt.tight_layout(); plt.show()

In [ ]:
# Scatter plots of each feature vs target
fig, axes = plt.subplots(1, min(4, len(num_feats)), figsize=(16, 4))
if len(num_feats) == 1: axes = [axes]
for ax, col in zip(axes, num_feats[:4]):
    ax.scatter(df[col], df[TARGET], alpha=0.2, s=5)
    ax.set_xlabel(col); ax.set_ylabel(TARGET)
    ax.set_title(f'{col} vs {TARGET}')
plt.tight_layout(); plt.show()

## 14. Target Analysis

Power output should follow a roughly normal distribution influenced by ambient temperature.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].hist(df[TARGET], bins=40, color='coral', alpha=0.7)
axes[0].set_title(f'{TARGET} Distribution (MW)')
axes[1].boxplot(df[TARGET].dropna(), vert=True)
axes[1].set_title(f'{TARGET} Box Plot')
plt.tight_layout(); plt.show()
print(f'Mean: {df[TARGET].mean():.2f} MW')
print(f'Median: {df[TARGET].median():.2f} MW')
print(f'Std: {df[TARGET].std():.2f} MW')
print(f'Skewness: {df[TARGET].skew():.2f}')

## 15. Train / Validation / Test Split

70/15/15 split with fixed seed.

In [ ]:
X = df.drop(columns=[TARGET])
y = df[TARGET]
X_tmp, X_test, y_tmp, y_test = train_test_split(X, y, test_size=TEST_SIZE, random_state=SEED)
X_train, X_val, y_train, y_val = train_test_split(X_tmp, y_tmp, test_size=VAL_SIZE/(1-TEST_SIZE), random_state=SEED)
print(f'Train: {X_train.shape}  Val: {X_val.shape}  Test: {X_test.shape}')

## 16. Preprocessing Strategy

- All 4 features are numeric — no categorical encoding needed
- Impute median + StandardScaler for linear models
- Tree models won't need scaling but it doesn't hurt

In [ ]:
num_cols = X_train.select_dtypes(include=[np.number]).columns.tolist()
preprocessor = ColumnTransformer([
    ('num', Pipeline([('imp', SimpleImputer(strategy='median')), ('sc', StandardScaler())]), num_cols)
], remainder='drop')
print(f'Numeric features: {len(num_cols)}')

## 17. Feature Engineering

- **AT²**: Quadratic temperature term for non-linear thermal effects
- **AT × V**: Interaction between temperature and vacuum
- **AT × RH**: Temperature-humidity interaction

In [ ]:
def engineer_features(d):
    d = d.copy()
    cols = d.columns.tolist()
    at_col = [c for c in cols if c.upper() in ['AT', 'TEMPERATURE', 'TEMP']]
    v_col = [c for c in cols if c.upper() in ['V', 'VACUUM']]
    rh_col = [c for c in cols if c.upper() in ['RH', 'HUMIDITY']]
    if at_col:
        at = at_col[0]
        d[f'{at}_squared'] = d[at] ** 2
        if v_col:
            d[f'{at}_x_{v_col[0]}'] = d[at] * d[v_col[0]]
        if rh_col:
            d[f'{at}_x_{rh_col[0]}'] = d[at] * d[rh_col[0]]
    return d

X_train = engineer_features(X_train)
X_val = engineer_features(X_val)
X_test = engineer_features(X_test)
num_cols = X_train.select_dtypes(include=[np.number]).columns.tolist()
preprocessor = ColumnTransformer([
    ('num', Pipeline([('imp', SimpleImputer(strategy='median')), ('sc', StandardScaler())]), num_cols)
], remainder='drop')
print(f'Features after engineering: {X_train.shape[1]}')

## 18. Baseline Model

Three baselines: DummyRegressor, LinearRegression, RandomForest.

In [ ]:
results = {}
for name, reg in [
    ('Dummy (mean)', DummyRegressor(strategy='mean')),
    ('LinearRegression', LinearRegression()),
    ('RandomForest', RandomForestRegressor(n_estimators=200, random_state=SEED, n_jobs=-1))
]:
    pipe = Pipeline([('pre', preprocessor), ('reg', reg)])
    pipe.fit(X_train, y_train)
    yp = pipe.predict(X_val)
    r = {'RMSE': np.sqrt(mean_squared_error(y_val, yp)),
         'MAE': mean_absolute_error(y_val, yp),
         'R2': r2_score(y_val, yp)}
    results[name] = r
    print(f"{name}: R2={r['R2']:.4f}, RMSE={r['RMSE']:.2f}, MAE={r['MAE']:.2f}")

## 19. LazyPredict Benchmark

In [ ]:
X_train_lp = preprocessor.fit_transform(X_train)
X_val_lp = preprocessor.transform(X_val)
lazy = LazyRegressor(verbose=0, ignore_warnings=True, random_state=SEED)
models_lp, _ = lazy.fit(X_train_lp, X_val_lp, y_train, y_val)
models_lp.head(15)

## 20. FLAML AutoML Run

In [ ]:
automl = AutoML()
automl.fit(X_train_lp, y_train, task='regression', metric='r2',
           time_budget=FLAML_BUDGET, seed=SEED, verbose=0)
print(f'Best estimator: {automl.best_estimator}')
yp_fl = automl.predict(X_val_lp)
results['FLAML'] = {'RMSE': np.sqrt(mean_squared_error(y_val, yp_fl)),
                    'MAE': mean_absolute_error(y_val, yp_fl),
                    'R2': r2_score(y_val, yp_fl)}
print(f"FLAML: R2={results['FLAML']['R2']:.4f}, RMSE={results['FLAML']['RMSE']:.2f}")

## 21. Top 3 Model Selection

In [ ]:
try:
    from lightgbm import LGBMRegressor; has_lgbm = True
except ImportError: has_lgbm = False
try:
    from xgboost import XGBRegressor; has_xgb = True
except ImportError: has_xgb = False
top3 = {}
if has_lgbm:
    top3['LightGBM'] = LGBMRegressor(n_estimators=500, learning_rate=0.05, max_depth=6, random_state=SEED, verbose=-1, n_jobs=-1)
else:
    from sklearn.ensemble import ExtraTreesRegressor
    top3['ExtraTrees'] = ExtraTreesRegressor(n_estimators=500, random_state=SEED, n_jobs=-1)
if has_xgb:
    top3['XGBoost'] = XGBRegressor(n_estimators=500, learning_rate=0.05, max_depth=6, random_state=SEED, verbosity=0, n_jobs=-1)
else:
    from sklearn.ensemble import AdaBoostRegressor
    top3['AdaBoost'] = AdaBoostRegressor(n_estimators=300, random_state=SEED)
top3['GradBoosting'] = GradientBoostingRegressor(n_estimators=400, learning_rate=0.05, max_depth=5, random_state=SEED)
print(f'Top 3: {list(top3.keys())}')

## 22. Final Training and Evaluation of Top 3

In [ ]:
X_tr_t = preprocessor.fit_transform(X_train)
X_te_t = preprocessor.transform(X_test)
final = {}
predictions = {}
for name, mdl in top3.items():
    mdl.fit(X_tr_t, y_train)
    yp = mdl.predict(X_te_t)
    predictions[name] = yp
    final[name] = {'RMSE': np.sqrt(mean_squared_error(y_test, yp)),
                   'MAE': mean_absolute_error(y_test, yp),
                   'R2': r2_score(y_test, yp)}
    print(f"{name}: R2={final[name]['R2']:.4f}, RMSE={final[name]['RMSE']:.2f} MW")
pd.DataFrame(final).T.sort_values('R2', ascending=False)

In [ ]:
fig, axes = plt.subplots(1, len(top3), figsize=(6*len(top3), 5))
if len(top3) == 1: axes = [axes]
for ax, (name, yp) in zip(axes, predictions.items()):
    ax.scatter(y_test, yp, alpha=0.3, s=10)
    mn, mx = min(y_test.min(), yp.min()), max(y_test.max(), yp.max())
    ax.plot([mn, mx], [mn, mx], 'r--', lw=2)
    ax.set_xlabel('Actual (MW)'); ax.set_ylabel('Predicted (MW)')
    ax.set_title(f"{name} (R2={final[name]['R2']:.3f})")
plt.tight_layout(); plt.show()

In [ ]:
fig, axes = plt.subplots(1, len(top3), figsize=(6*len(top3), 5))
if len(top3) == 1: axes = [axes]
for ax, (name, yp) in zip(axes, predictions.items()):
    residuals = y_test.values - yp
    ax.scatter(yp, residuals, alpha=0.3, s=10)
    ax.axhline(0, color='red', linestyle='--')
    ax.set_xlabel('Predicted'); ax.set_ylabel('Residual')
    ax.set_title(f'{name} Residuals')
plt.tight_layout(); plt.show()

## 23. Error Analysis

In [ ]:
best_name = max(final, key=lambda k: final[k]['R2'])
best_preds = predictions[best_name]
abs_errors = np.abs(y_test.values - best_preds)
print(f'Best model: {best_name}')
print(f'Mean absolute error: {abs_errors.mean():.2f} MW')
print(f'Median absolute error: {np.median(abs_errors):.2f} MW')
print(f'90th percentile error: {np.percentile(abs_errors, 90):.2f} MW')

## 24. Interpretation and Business Insight

- **Ambient Temperature (AT)** is the strongest predictor — higher temperatures reduce output
- **Exhaust Vacuum (V)** has a strong positive correlation with output
- **Relative Humidity (RH)** has a mild negative effect
- **Ambient Pressure (AP)** has a weak positive relationship
- The physics is clear: gas turbines are less efficient in hot weather
- Linear regression already explains ~93% of variance; boosting improves the non-linear tail

## 25. Limitations

1. Only 4 ambient variables — no plant operational parameters
2. Data from a single plant
3. No temporal structure (load scheduling, maintenance)
4. No fuel quality or turbine degradation info
5. Hourly aggregation may mask sub-hourly dynamics

## 26. How to Improve This Project

1. Add operational parameters (fuel flow, turbine speed)
2. Include time features for seasonal patterns
3. Multi-plant generalization study
4. Physics-informed ML (embed thermodynamic constraints)
5. Quantile regression for prediction intervals

## 27. Production Considerations

- Real-time output forecasting for dispatch optimization
- Weather API integration for day-ahead predictions
- Anomaly detection for plant performance degradation
- Model retraining as turbine ages
- Dashboard for plant operators

## 28. Common Mistakes

1. Not understanding the inverse AT–PE relationship
2. Ignoring feature interactions (AT × V)
3. Overfitting with too-complex models on simple data
4. Not checking for temporal patterns in residuals
5. Using the dataset for time-series without time ordering

## 29. Mini Challenge / Exercises

1. Compare polynomial regression (degree 2, 3) vs tree models
2. Try SVR with RBF kernel
3. Use 10-fold CV and report mean ± std R²
4. Identify which feature contributes most using permutation importance
5. Try predicting residuals from the linear model with a tree model (stacking)

## 30. Final Summary / Key Takeaways

| Aspect | Detail |
|---|---|
| Task | Regression — predict power plant output |
| Dataset | UCI CCPP (9,568 records, 4 features) |
| Difficulty | Easy |
| Key insight | Temperature dominates; physics is clear |
| Best approach | Gradient boosting captures non-linear thermal effects |
| Primary metric | R² |
| Baseline comparison | Even linear regression achieves high R² |